# 01 — Without A2A: The Pain of Custom Agent Integration

## Why this notebook exists

Imagine you have two services that need to collaborate:

- A **researcher** that gathers facts on a topic.
- A **writer** that turns those facts into a paragraph.

In a world without a standard agent-to-agent protocol, the only way to connect them is to design a custom REST API for each service, then hand-roll the glue code. That works — until one side changes its contract. Then the glue breaks, and every other consumer of that service breaks with it.

This notebook reproduces that pain in ~100 lines of code. Once you've felt it, the rest of the series will introduce **A2A** — a standard that makes this problem mostly go away.

## What you'll learn

- How to stand up two FastAPI services in a notebook and call them with `httpx`.
- Why ad-hoc REST contracts between agent-like services don't scale.
- What a "breaking change" looks like in practice, and why coordination is expensive without a shared protocol.
- The motivation for the rest of this series: **A2A** as a standard for agent-to-agent communication.

## 1. Setup

We'll use:

- **FastAPI** to define each service.
- **uvicorn** running on a background thread so we can both serve and call the API from the same notebook.
- **httpx** as our HTTP client.
- **pydantic** for typed request/response models.

The helper `run_server_in_thread(app, port)` starts a uvicorn server in a daemon thread and returns a handle we can use to shut it down at the end of the notebook.

In [1]:
import socket
import threading
import time

import httpx
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    """Start `app` on localhost:`port` in a background daemon thread.

    Returns the uvicorn Server object so the caller can stop it later.
    """
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")

    _servers.append(server)
    return server


def _port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def stop_server(server: uvicorn.Server, port: int) -> None:
    """Stop a single server and wait for its port to be released."""
    server.should_exit = True
    for _ in range(50):
        if _port_is_free(port):
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Port {port} still bound after stop")
    if server in _servers:
        _servers.remove(server)


def shutdown_all_servers() -> None:
    """Stop every server we started in this notebook."""
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")

Setup OK


## 2. The Researcher Service

The researcher takes a `topic` and returns a list of "facts" about it. We're not using a real LLM — this is a stub that returns canned facts, because the point of this notebook is the **integration**, not the intelligence.

The researcher's API contract: `POST /research` with `{"topic": str}` → `{"topic": str, "facts": list[str]}`.

In [2]:
researcher_app = FastAPI()


class ResearchRequest(BaseModel):
    topic: str


class ResearchResponse(BaseModel):
    topic: str
    facts: list[str]


FACTS_BY_TOPIC = {
    "octopuses": [
        "Octopuses have three hearts.",
        "They can change color in under a second.",
        "Each of their arms has its own neural cluster.",
    ],
    "rome": [
        "Rome was founded in 753 BCE according to tradition.",
        "The Roman Empire at its peak spanned roughly 5 million km².",
        "Roman concrete used volcanic ash and is still studied today.",
    ],
}


@researcher_app.post("/research", response_model=ResearchResponse)
def research(req: ResearchRequest) -> ResearchResponse:
    facts = FACTS_BY_TOPIC.get(req.topic.lower())
    if facts is None:
        raise HTTPException(status_code=404, detail=f"No facts on file for {req.topic!r}")
    return ResearchResponse(topic=req.topic, facts=facts)


researcher_server = run_server_in_thread(researcher_app, port=8010)
print("Researcher running on http://127.0.0.1:8010")

Researcher running on http://127.0.0.1:8010


In [3]:
resp = httpx.post("http://127.0.0.1:8010/research", json={"topic": "octopuses"})
print(resp.status_code)
print(resp.json())

200
{'topic': 'octopuses', 'facts': ['Octopuses have three hearts.', 'They can change color in under a second.', 'Each of their arms has its own neural cluster.']}


## 3. The Writer Service

The writer takes a `topic` and a list of `facts` and returns a one-paragraph summary. Like the researcher, it's a stub — it just joins the facts together with a templated lead-in.

The writer's API contract: `POST /write` with `{"topic": str, "facts": list[str]}` → `{"paragraph": str}`.

Note how this contract was designed **independently** of the researcher's. The researcher returns `{"topic", "facts"}`; the writer expects `{"topic", "facts"}`. The fact that they happen to line up is a coincidence we (the integrator) have to maintain by hand.

In [4]:
writer_app = FastAPI()


class WriteRequest(BaseModel):
    topic: str
    facts: list[str]


class WriteResponse(BaseModel):
    paragraph: str


@writer_app.post("/write", response_model=WriteResponse)
def write(req: WriteRequest) -> WriteResponse:
    bullets = " ".join(req.facts)
    paragraph = f"Here is what we know about {req.topic}: {bullets}"
    return WriteResponse(paragraph=paragraph)


writer_server = run_server_in_thread(writer_app, port=8011)
print("Writer running on http://127.0.0.1:8011")

Writer running on http://127.0.0.1:8011


In [5]:
resp = httpx.post(
    "http://127.0.0.1:8011/write",
    json={"topic": "octopuses", "facts": ["They have three hearts.", "They can change color."]},
)
print(resp.status_code)
print(resp.json())

200
{'paragraph': 'Here is what we know about octopuses: They have three hearts. They can change color.'}


## 4. Hand-Rolled Integration

Now the glue: a function `research_and_write(topic)` that calls the researcher, then forwards its output to the writer, then returns the paragraph.

Read this code carefully. Notice:

- We have to **know the URLs** of both services.
- We have to **know the request/response shapes** of both services.
- We have to **manually translate** between them (in this case, the shapes match exactly — but only because we got lucky).
- There is **no machine-readable description** of either service. The only way another engineer learns what's available is by reading our code, the services' source, or whatever docs we happen to write.

In [6]:
RESEARCHER_URL = "http://127.0.0.1:8010"
WRITER_URL = "http://127.0.0.1:8011"


def research_and_write(topic: str) -> str:
    research_resp = httpx.post(f"{RESEARCHER_URL}/research", json={"topic": topic})
    research_resp.raise_for_status()
    research_data = research_resp.json()

    write_resp = httpx.post(
        f"{WRITER_URL}/write",
        json={"topic": research_data["topic"], "facts": research_data["facts"]},
    )
    write_resp.raise_for_status()
    write_data = write_resp.json()

    return write_data["paragraph"]


print(research_and_write("octopuses"))

Here is what we know about octopuses: Octopuses have three hearts. They can change color in under a second. Each of their arms has its own neural cluster.


## 5. The Breakage

Now imagine the writer team decides their API should be more "RESTful". They rename the request field `facts` to `bullet_points` and ship a new version. They tell their team in Slack. They don't tell us.

Watch what happens.

In [7]:
# Simulate the writer team shipping a breaking change:
# stop the old writer, define a new app with a renamed field, start it on the same port.

stop_server(writer_server, port=8011)

writer_app_v2 = FastAPI()


class WriteRequestV2(BaseModel):
    topic: str
    bullet_points: list[str]   # was: facts


@writer_app_v2.post("/write", response_model=WriteResponse)
def write_v2(req: WriteRequestV2) -> WriteResponse:
    bullets = " ".join(req.bullet_points)
    paragraph = f"Here is what we know about {req.topic}: {bullets}"
    return WriteResponse(paragraph=paragraph)


writer_server = run_server_in_thread(writer_app_v2, port=8011)
print("Writer v2 running on http://127.0.0.1:8011 — now expects 'bullet_points' instead of 'facts'")

Writer v2 running on http://127.0.0.1:8011 — now expects 'bullet_points' instead of 'facts'


In [8]:
try:
    print(research_and_write("rome"))
except httpx.HTTPStatusError as e:
    print(f"BROKE: {e.response.status_code}")
    print(e.response.json())

BROKE: 422
{'detail': [{'type': 'missing', 'loc': ['body', 'bullet_points'], 'msg': 'Field required', 'input': {'topic': 'rome', 'facts': ['Rome was founded in 753 BCE according to tradition.', 'The Roman Empire at its peak spanned roughly 5 million km².', 'Roman concrete used volcanic ash and is still studied today.']}}]}


### What just happened?

The writer's HTTP server is still up. The researcher is still up. But the integration code is broken because:

- The writer changed its input schema.
- The integration code had no way to know.
- There was no shared description of the writer's contract that we could check against.
- There was no version negotiation, no capability discovery, no nothing.

Now imagine you're not running two services. Imagine you're running twenty, written by five different teams. Every integration is bespoke. Every breaking change is a multi-team coordination problem.

**This is the problem A2A is designed to solve.**

## What you just learned

- How to stand up two FastAPI services side-by-side in a notebook.
- What hand-rolled integration code looks like when there's no shared agent protocol.
- How a one-field rename on one service breaks every consumer that hard-coded the old contract.
- That coordinating breaking changes across multiple agent services is **the** problem to solve.

## What's missing

We have no machine-readable description of either service. A consumer can't ask "what skills do you have, what inputs do you take, and what authentication do you require?" and get a structured answer.

In **notebook 02**, we introduce the first piece of A2A: the **Agent Card**, served at `/.well-known/agent.json`. The card is a JSON document that answers exactly that question, and it's the entry point for everything else in the protocol.

In [9]:
shutdown_all_servers()
print("All servers stopped.")

All servers stopped.
